In [ ]:
from sympy import *
%run Geom_Prolongation.ipynb
%run Particular_Distributions.ipynb

In [ ]:
# ~ 15 hrs through weight 9
g=Symp_symb(7)
E=g.ext_alg
C=g.cochain_complex
D,D_JS=Standard_Prenorm_distr(g,6)
P=Geom_Prolongation(D,D_JS)
y,h,e=symbols('y,h,e')
K=IndexedBase('K')
Y,H,E,X,e1,e2,e3,e4,e5,e6,N=g.basis

In [ ]:
def check_equivar(c):
    p_inv=-exp(-2*h)*y*Y-h*H-e*E
    c_0=c.subs({y:0,h:0,e:0})
    r=p_inv.Ad(c_0,mod='CE')-c
    simplify_cochain(r)
    if r==C.elt({}): return True
    else: return r

In [ ]:
F1,K1=P.normal_frame(1)
simplify_cochain(K1)
print('K1 =',K1)

In [ ]:
F2,K2=P.normal_frame(2)
simplify_cochain(K2)
print('K2 =',K2)

In [ ]:
F3,K3=P.normal_frame(3)
simplify_cochain(K3)

In [ ]:
F4,K4=P.normal_frame(4)
simplify_cochain(K4)

In [ ]:
F5,K5=P.normal_frame(5)

In [ ]:
F6,K6=P.normal_frame(6)

In [ ]:
K3_subbed=ds_subs(K3,D_JS,D)
K4_subbed=ds_subs(K4,D_JS,D)
K5_subbed=ds_subs(K5,D_JS,D)
K6_subbed=ds_subs(K6,D_JS,D)

In [ ]:
K3h=simplify(C.subspace_proj(K3_subbed,'harmonic').symb_expr())
K4h=simplify(C.subspace_proj(K4_subbed,'harmonic').symb_expr()-K3h)
K5h=simplify(C.subspace_proj(K5_subbed,'harmonic').symb_expr()-K3h-K4h)
K6h=simplify(C.subspace_proj(K6_subbed,'harmonic').symb_expr()-K3h-K4h-K5h)

In [ ]:
temp_dict={}
for w in C.basis(2):
    for c in C.basis(2,w):
        temp_dict[c.symb_expr()]=0
temp_dict[symbols('e')]=0
temp_dict[symbols('h')]=0

temp_dict[symbols('{{e_1^*\wedge}e_4^*\otimes}e_2')]=Rational(1,3)
temp_dict[symbols('{{X^*\wedge}e_4^*\otimes}e_1')]=Rational(1,5)
temp_dict[symbols('{{X^*\wedge}e_6^*\otimes}e_1')]=1

K3h_coeff=K3h.subs(temp_dict)
K4h_coeff=K4h.subs(temp_dict)
K6h_coeff=K6h.subs(temp_dict)

In [ ]:
display(K3h_coeff)
display(K4h_coeff)
display(K6h_coeff)

## Syzygies

In [ ]:
from itertools import combinations_with_replacement
from itertools import product
I=IndexedBase('I')
W=IndexedBase('W')
Abstract_Invars={} # keys are weights; values are abstract invariants
Concrete_Invars={I[3]:K3h_coeff,W[4]:K4h_coeff,W[6]:K6h_coeff} # keys are homogeneous invar monomials; vals are the corresponding concrete invars
Harm_Invars={} # keys are weights; values are (abstract_invar,concrete_invar)
Invar_Mats={} # keys are weights; values are mats with cols giving the coords of concrete invars in the concrete monomial basis
Invar_nullspaces={} # keys are weights; values are the nullspaces of the above matrices
Concrete_monom_basis={} # keys are weights; values are lists of monomials which span the concrete invariants in the weight

fundamental_invars=[I[3],W[4],W[6]]

In [ ]:
def coordinatize_in_monom_basis(expr1,basis):
    """Returns the coordinates of expr in the given basis as a list
    INPUTS:
    * 'expr' - a linear combination of the monomials from basis
    * 'basis' - a list of monomials in symbols and Indexed objects"""
    r=[0]*len(basis)
    expr=expand(expr1)
    for A in expr.as_coeff_add()[1]:
        try: r[basis.index(prod(A.as_coeff_mul()[1]))]=A.as_coeff_mul()[0]
        except ValueError: 
            print('coordinatize_in_monom_basis received expr not in span(basis)')
            print('expr =',expr)
            print('basis =',basis)
            return None
    return r

def monomials_in_expr(expr1):
    """Returns the set of all monomials involved in expr, which should be a polynomial over R
    INPUTS:
    * 'expr' - a polynomial in symbols and Indexed objects over R"""
    r=set()
    expr=expand(expr1)
    if isinstance(expr,numbers.Number): return r
    if type(expr)==Add:
        for A in expr.args:
            r=r.union(monomials_in_expr(A))
    if type(expr) in [Mul, Pow]:
        r.add(expr.as_coeff_Mul()[1])
    if type(expr) in [Indexed,Symbol]:
        r.add(expr)
    return r

def higher_der(expr,*ders):
    r=expr
    for i in ders:
        r=D.abn_ind_der(r,i)
    return r

def coeff_add_to_monom_dict(ca):
    r={}
    for A in ca:
        r[A.as_coeff_Mul()[1]]=A.as_coeff_Mul()[0]
    return r

def common_monom(d1,d2):
    return set(d1.keys()).intersection(set(d2.keys()))

In [ ]:
def all_partitions(s,Z):
    """Returns the set of all tuples (d1,d2,...dj) so that z1*d1 + ... + zj*dj=s
    where Z={z1,z2,...zj}
    INPUTS:
    * 's' - a natural number
    * 'Z' - a finite list of distinct natural numbers (without 0)"""

    if s==0: return {tuple([0]*len(Z))}
    if s<0: return set()
    if len(Z)==0:
        return set()
    
    r=set()
    for a in range(s//Z[0]+1):
        new_Z=copy.copy(Z)
        new_Z.remove(Z[0])
        r=r.union({tuple([a]+list(A)) for A in all_partitions(s-a*Z[0],new_Z)})
    return r

def multiply_cart_prod_elts(L):
    r1,r2=1,1
    for a in L:
        a1=[b[0] for b in a]
        a2=[b[1] for b in a]
        r1=r1*prod(a1)
        r2=r2*prod(a2)
    return (r1,r2)

In [ ]:
def Compute_abstract_invars(wght,additional_invars,reshelve=False,unshelve=False):
    """Adds invariants to Abstract_Invars of the given weight by taking derivatives and
    products of previous abstract invariants and by adding the given set of additional invariants.
    Assumes invariants of lower weights have already been computed.
    INPUTS:
    * 'wght' - a natural number weight
    * 'additional_invar' - a set of invariants as pairs (Indexed,expr) (for example {(W[4],5*K[3,9,6]/18)})
    """
    if unshelve:
        try:
            with shelve.open('harm_invars') as shelf:
                Abstract_Invars[wght]=shelf['Standard_norm_Abstract_Invars'+'{j}'.format(j=wght)]
                return None
        except: pass

    # additional invariants
    Abs_candidates={A for A in additional_invars}

    # derivatives of previous invariants
    if wght-1 in Abstract_Invars:
        HI=list(Abstract_Invars[wght-1])
        for invar in HI:
            Abs_candidates=Abs_candidates.union(monomials_in_expr(D.abn_ind_der(invar,3)))
            Abs_candidates=Abs_candidates.union(monomials_in_expr(D.abn_ind_der(invar,4)))
    
    # products of previous invariants
    # The below block is wrong!
    for p in all_partitions(wght,list(range(3,wght))):
        big_list=[list(combinations_with_replacement(Abstract_Invars[list(range(3,wght))[i]],p[i])) for i in range(len(p))]
        p_prods=list(itertools.product(*big_list))
        for pp in p_prods:
            Abs_candidates.add(prod([prod(a) for a in pp]))
    
    Abs_candidates=list(Abs_candidates)

    temp_basis=set()
    for invar in Abs_candidates:
        temp_basis=temp_basis.union(monomials_in_expr(invar))
    temp_basis=list(temp_basis)

    A=zeros(len(temp_basis),0)
    for invar in Abs_candidates:
        A=A.col_insert(shape(A)[1],Matrix(coordinatize_in_monom_basis(invar,temp_basis)))
    pivs=A.rref()[1]
    Abstract_Invars[wght]=[Abs_candidates[i] for i in pivs]

    if reshelve:
        try:
            with shelve.open('harm_invars') as shelf:
                shelf['Standard_norm_Abstract_Invars'+'{j}'.format(j=wght)]=Abstract_Invars[wght]
        except: print('Reshelving of harm_invars in wght',wght,'Failed')

In [ ]:
def Compute_concrete_invars(wght,conc_dict,reshelve=False,unshelve=False):
    """Converts the abstract invariants of weight wght to concrete invariants,
    storing the results in conc_dict, assuming Abstract_Invars[wght] has been computed
    
    WARNING: Unshelving here isn't subdivided by weight, so whatever weight was shelved will be restored,
    overwriting those weights already computed."""
    if unshelve:
        with shelve.open('harm_invars') as shelf:
                conc_dict=shelf['Standard_norm_Concrete_Invars']
                return None
                # for invar in shelf['Standard_norm_conc_dict']:
                #     conc_dict[invar]=shelf['Standard_norm_Concrete_Invars'][Invar]

    for invar in Abstract_Invars[wght]:
        if invar not in fundamental_invars:
            if len(invar.as_coeff_mul()[1])>1:
                temp_subs={a:conc_dict[a] for a in invar.as_coeff_mul()[1]}
                r=invar.xreplace(temp_subs)
            elif type(invar.as_coeff_mul()[1][0])==Pow:
                temp_subs={invar.as_coeff_mul()[1][0].as_base_exp()[0]:conc_dict[invar.as_coeff_mul()[1][0].as_base_exp()[0]]}
                r=invar.xreplace(temp_subs)
            else:
                r=ds_subs(D.abn_ind_der(conc_dict[invar.base[invar.indices[0:-1]]],invar.indices[-1]),D_JS,D)
            conc_dict[invar]=r
    if reshelve:
        try:
            with shelve.open('harm_invars') as shelf:
                shelf['Standard_norm_Concrete_Invars']=conc_dict
        except: pass

In [ ]:
Abstract_Invars={}
Compute_abstract_invars(3,{I[3]},unshelve=True)
Compute_abstract_invars(4,{W[4]},unshelve=True)
Compute_abstract_invars(5,set(),unshelve=True)
Compute_abstract_invars(6,{W[6]},unshelve=True)
Compute_abstract_invars(7,set(),unshelve=True)
Compute_abstract_invars(8,set(),unshelve=True)
Compute_abstract_invars(9,set(),unshelve=True)

In [ ]:
# 45 min to weight 9
time0=time.time()
Compute_concrete_invars(3,Concrete_Invars)
Compute_concrete_invars(4,Concrete_Invars)
Compute_concrete_invars(5,Concrete_Invars)
Compute_concrete_invars(6,Concrete_Invars)
Compute_concrete_invars(7,Concrete_Invars)
time1=time.time()
print('Weights 1-7 computed in time',hrs_min_sec(time1-time0))
Compute_concrete_invars(8,Concrete_Invars)
time2=time.time()
print('Weight 8 computed in time',hrs_min_sec(time2-time1)) # 2 min 30 sec up to here

### Manual Attempts

In [ ]:
def Compute_concrete_invars_manual(wght,conc_dict,monom_set):
    """Converts the abstract invariants of weight wght to concrete invariants,
    storing the results in conc_dict if any monomials from monom_set are involved in the result
    
    WARNING: Unshelving here isn't subdivided by weight, so whatever weight was shelved will be restored,
    overwriting those weights already computed."""
    print('Total invars to process:',len(Abstract_Invars[wght]))
    cnt=1
    for invar in Abstract_Invars[wght]:
        if invar not in fundamental_invars:
            time0=time.time()
            if len(invar.as_coeff_mul()[1])>1:
                print('type 1')
                temp_subs={a:conc_dict[a] for a in invar.as_coeff_mul()[1]}
                r=invar.xreplace(temp_subs)
            elif type(invar.as_coeff_mul()[1][0])==Pow:
                print('type 2')
                temp_subs={invar.as_coeff_mul()[1][0].as_base_exp()[0]:conc_dict[invar.as_coeff_mul()[1][0].as_base_exp()[0]]}
                r=invar.xreplace(temp_subs)
            else:
                print('type 3')
                temp1=collect_some(conc_dict[invar.base[invar.indices[0:-1]]])
                print('temp1 =',temp1)
                temp2=D.abn_ind_der(temp1,invar.indices[-1])
                print('temp2 =',temp2)
                r=ds_subs(temp2,D_JS,D)
            if monomials_in_expr(expand(r)).intersection(monom_set)!=set(): conc_dict[invar]=r

            # temporary for testing:
            print(invar,'===>',r)
            print(monomials_in_expr(expand(r)))
        print('invar',cnt,'processed in time',hrs_min_sec(time.time()-time0))

In [ ]:
## collect_some might speed up the process of taking abnormal derivatives

import heapq

def incr_key(d,k):
    """d-dict, k-key; increments value at k or makes d[k]=1"""
    if k in d:
        d[k]=d[k]+1
    else: d[k]=1

def count_terms(expr):
    """A heuristic count of what terms show up a lot in the polynomial expr"""
    r={}
    count_terms_rec(expr,r)
    return r

def count_terms_rec(expr,count_dict):
    if isinstance(expr,numbers.Number): return None
    if type(expr) in [Add,Mul]:
        for a in expr.args: count_terms_rec(a,count_dict)
    if type(expr)==Pow: count_terms_rec(expr.base,count_dict)
    if type(expr) in [Indexed,Symbol]: incr_key(count_dict,expr)

def top_k_keys(d, k):
    if k <= 0:
        return []

    # Create a min-heap to store the top k keys, including their original order
    min_heap = []

    for index, (key, value) in enumerate(d.items()):
        # Push a tuple of (value, original index, key) onto the heap
        heapq.heappush(min_heap, (value, index, key))
        # If the heap size exceeds k, pop the smallest element
        if len(min_heap) > k:
            heapq.heappop(min_heap)

    # Extract the keys from the heap, sorting them by values and original index
    top_keys = [key for _, _, key in sorted(min_heap, key=lambda x: (-x[0], x[1]))]
    
    return top_keys

def collect_some(expr):
    """Chooses some frequently occurring terms in the polynomial expr and collects those terms"""
    ## Count occurrences
    terms=count_terms(expr)
    if terms=={}: return copy.copy(expr)
    total_terms = sum(terms.values()) # use this to decide how many terms to collect
    num_terms=log(total_terms,10) # This is an arbitrary choice
    collect_terms=top_k_keys(terms,num_terms)

    r=copy.copy(expr)
    collect(r,collect_terms,order=None)
    return r

In [ ]:
I3_monoms=monomials_in_expr(Concrete_Invars[I[3]])

In [ ]:
temp=Concrete_Invars[I[3, 4, 3, 3, 4, 4]]

In [ ]:
temp_terms=count_terms(temp)
log(sum(temp_terms.values()),10).evalf()

In [ ]:
collect(temp,K[5,8,9])

In [ ]:
collect(temp,top_k_keys(temp_terms,4))

In [ ]:
temp

In [ ]:
temp1=collect_some(temp)

In [ ]:
D.abn_ind_der(temp,4)

In [ ]:
temp1=collect(temp,[K[3, 9, 6, 4, 4, 4],K[3, 9, 6, 4, 4],K[3, 9, 6, 4],K[3, 9, 6]])

In [ ]:
temp2=D.abn_ind_der(temp1,4)

In [ ]:
# This suggests the derivatives are actually what's taking a long time...surprising
Compute_concrete_invars_manual(9,Concrete_Invars,I3_monoms)

In [ ]:
abstract_t=[]
t=[]
t.append(expand(ds_subs(K3h_coeff**3,D_JS,D)))
abstract_t.append(('Type 0'))

type_count=0

# # Very slow!
# P=list(product([3,4],[3,4],[3,4],[3,4],[3,4],[3,4]))
# print(len(P))
# for i in range(len(P)):
#     print(i)
#     t.append(expand(ds_subs(higher_der(K3h_coeff,*(P[i])),D_JS,D)))
#     abstract_t.append(('Type 2',P[i]))

# # # --------------------------------------------

# Derivatives of W4
P=list(product([3,4],[3,4],[3,4],[3,4],[3,4]))
for i in range(len(P)):
    t.append(expand(ds_subs(higher_der(K4h_coeff,*(P[i])),D_JS,D)))
    abstract_t.append(('Type'+str(type_count),P[i]))
type_count+=1

# # # --------------------------------------------
# (Derivatives of W4)*W3
P=list(product([3,4],[3,4]))
for i in range(len(P)):
    t.append(expand(ds_subs(higher_der(K4h_coeff,*P[i]),D_JS,D)*K3h_coeff))
    abstract_t.append(('Type'+str(type_count),P[i]))
type_count+=1

# (derivatives of W4)*(derivatives of W3)
P=list(product([3,4],[3,4]))
for i in range(len(P)):
    t.append(expand(ds_subs(higher_der(K3h_coeff,P[i][0]),D_JS,D)*ds_subs(higher_der(K4h_coeff,P[i][1]),D_JS,D)))
    abstract_t.append(('Type'+str(type_count),P[i]))
type_count+=1

# W4*(derivatives of W3)
P=list(product([3,4],[3,4]))
for i in range(len(P)):
    t.append(expand(ds_subs(higher_der(K3h_coeff,*P[i]),D_JS,D)*K4h_coeff))
    abstract_t.append(('Type'+str(type_count),P[i]))
type_count+=1

# # # --------------------------------------------

# W6*W3
t.append(expand(K6h_coeff*K3h_coeff))
abstract_t.append(('Type'+str(type_count)))
type_count+=1


# Derivatives of W6
P=list(product([3,4],[3,4],[3,4]))
for i in range(len(P)):
    t.append(expand(ds_subs(higher_der(K6h_coeff,*(P[i])),D_JS,D)))
    abstract_t.append(('Type'+str(type_count),P[i]))
type_count+=1



d=[coeff_add_to_monom_dict(A.as_coeff_add()[1]) for A in t]

In [ ]:
da=coeff_add_to_monom_dict(expand(Concrete_Invars[I[3]**3]).as_coeff_add()[1])
common_mon_dict={}
ctr=1
for k in Abstract_Invars[9]:
    db=coeff_add_to_monom_dict(expand(Concrete_Invars[k]).as_coeff_add()[1])
    common_mon_dict[k]=common_monom(da,db)
    print('Invar',ctr,'=',k,'complete in time',hrs_min_sec(time.time()))

In [ ]:
print(common_mon_dict)

In [ ]:
da=d[0]
for db in d[1:len(d)]:
    if common_monom(da,db)!=set():
        counter={}
        for A in set(da.keys()).intersection(set(db.keys())):
            temp=da[A]/db[A]
            if temp in counter: counter[temp]=counter[temp]+1
            else: counter[temp]=1

        for key in counter:
            if counter[key]>1: print(key,'-->',counter[key])

### Systematic Attempts

#### Compute All (Abstact) Invariant Monomials

#### Convert Abstract Invariant Monomials into Concrete Invariants

In [ ]:
# Unshelving Concrete_Invars
Compute_concrete_invars(0,Concrete_Invars,unshelve=True)

In [ ]:
# 45 min
time0=time.time()
Compute_concrete_invars(3,Concrete_Invars)
Compute_concrete_invars(4,Concrete_Invars)
Compute_concrete_invars(5,Concrete_Invars)
Compute_concrete_invars(6,Concrete_Invars)
Compute_concrete_invars(7,Concrete_Invars)
time1=time.time()
print('Weights 1-7 computed in time',hrs_min_sec(time1-time0))
Compute_concrete_invars(8,Concrete_Invars)
time2=time.time()
print('Weight 8 computed in time',hrs_min_sec(time2-time1)) # 2 min 30 sec up to here
Compute_concrete_invars(9,Concrete_Invars,reshelve=False)
print('Weight 9 computed in time',hrs_min_sec(time.time()-time2))

#### Compute Concrete Monomial bases

In [ ]:
def Compute_concrete_monom_basis(wght,reshelve=False,unshelve=False):
    """Computes a basis of monomials spanning the concrete invariants in wght, storing
    the results in Concrete_monom_basis"""
    if unshelve:
        try:
            with shelve.open('harm_invars') as shelf:
                Concrete_monom_basis[wght]=shelf['Standard_norm_Concrete_monom_basis'+'{j}'.format(j=wght)]
                return None
        except: pass

    result=set()
    for A in Abstract_Invars[wght]:
        result=result.union(monomials_in_expr(Concrete_Invars[A]))
    Concrete_monom_basis[wght]=list(result)

    if reshelve:
        try:
            with shelve.open('harm_invars') as shelf:
                shelf['Standard_norm_Concrete_monom_basis'+'{j}'.format(j=wght)]=Concrete_monom_basis[wght]
        except: print('Reshelving of Concrete_monom_basis in wght',wght,'Failed')

In [ ]:
# 3 min 30 sec thru weight 8
for i in range(3,10):
    Compute_concrete_monom_basis(i,unshelve=True)

#### Coordinatize, Find Syzygies

In [ ]:
def compute_invar_mat(wght,reshelve=False,unshelve=False):
    """Sets the matrix representation of all invariants of given weight as the value Invar_Mats[wght].
    Invar_Mats[wght] is a Matrix with columns given by the Concrete Invariants of weight wght as column i,
    represented in Concrete_monom_basis[wght]
    
    Assumes Concrete_monom_basis[wght] has been computed"""

    if unshelve:
        try:
            with shelve.open('Syzygies') as shelf:
                Invar_Mats[wght]=shelf['Standard_norm_invar_mats'+'{j}'.format(j=wght)]
                return None
        except: pass

    Invar_Mats[wght]=zeros(len(Concrete_monom_basis[wght]),0)
    for A in Abstract_Invars[wght]:
        temp_col=Matrix([[a] for a in coordinatize_in_monom_basis(Concrete_Invars[A],Concrete_monom_basis[wght])])
        Invar_Mats[wght]=Invar_Mats[wght].col_insert(shape(Invar_Mats[wght])[1],temp_col)

    if reshelve:
        try:
            with shelve.open('Syzygies') as shelf:
                shelf['Standard_norm_invar_mats'+'{j}'.format(j=wght)]=Invar_Mats[wght]
        except: print('Reshelving of invar_mats in wght',wght,'Failed')

def compute_Invar_nullspace(wght,reshelve=False,unshelve=False):
    """Assuming compute_invar_mat(wght) has been run, this computes the nullspace of that matrix
     (i.e., the relations between the invariants in the given wght, as columns)"""
    
    if unshelve:
        try:
            with shelve.open('Syzygies') as shelf:
                Invar_nullspaces[wght]=shelf['Standard_norm_invar_nullspaces'+'{j}'.format(j=wght)]
                return None
        except: pass

    Invar_nullspaces[wght]=Invar_Mats[wght].nullspace()

    if reshelve:
        try:
            with shelve.open('Syzygies') as shelf:
                shelf['Standard_norm_invar_nullspaces'+'{j}'.format(j=wght)]=Invar_nullspaces[wght]
        except: print('Reshelving of invar_nullspaces in wght',wght,'Failed')

In [ ]:
# def K_wght(expr):
#     """Given a monomial in the IndexedBase K, returns the weight of """
#     if type(expr)==Add:
#         if expr.as_coeff_add()[0]!=0: r=0
#         else: r=K_wght(expr.as_coeff_add()[1][0])
#         for A in expr.as_coeff_add()[1]:
#             if K_wght(A)!=r:
#                 return None
#         return r
#     if type(expr)==Mul:
#         return sum([K_wght(A) for A in expr.as_coeff_mul()[1]])
#     if type(expr)==Pow:
#         return expr.as_base_exp()[1]*K_wght(expr.as_base_exp()[0])
#     if type(expr)==Rational:
#         return 0
#     if type(expr)==Indexed:
#         r=-g.basis[expr.indices[0]].wght-g.basis[expr.indices[1]].wght+g.basis[expr.indices[2]].wght
#         for i in expr.indices[3:len(expr.indices)]:
#             r=r-g.basis[i].wght
#         return r

In [ ]:
# def Abstract_to_Concrete_Invar(expr):
#     temp_subs={}
#     I_set=Indexed_obj_in_expr(expr)
#     for A in I_set:
#         for wght in Abstract_Invars_vecs:
#             if A in Abstract_Invars_vecs[wght]:
#                 i=list(Abstract_Invars_vecs[wght]).index(A)
#                 w=wght
#         temp_subs[A]=Concrete_Invars_vecs[w][i]
#     return expr.xreplace(temp_subs)

In [ ]:
for wght in Abstract_Invars:
    print(wght,'-->',len(Abstract_Invars[wght]))

print('\n')
for wght in Concrete_monom_basis:
    print(wght,'-->',len(Concrete_monom_basis[wght]))

In [ ]:
# 3 min thru weight 8
for wght in range(3,10):
    time0=time.time()
    compute_invar_mat(wght,unshelve=True)
    print('Invar_mat of weight',wght,'computed in time',hrs_min_sec(time.time()-time0))

In [ ]:
# 1 min thru weight 8
for wght in range(3,9):
    time1=time.time()
    compute_Invar_nullspace(wght,unshelve=True)
    print('nullspace of weight',wght,'computed in time',hrs_min_sec(time.time()-time1),'\n')

In [ ]:
shape(Invar_Mats[9]) 

In [ ]:
i=shape(Invar_Mats[9])[1]
A=Invar_Mats[9][0:i,0:i]
A_null=A.nullspace()

In [ ]:
with shelve.open('Syzygies') as shelf:
    shelf['A_null']=A_null

In [ ]:
for wght in range(9,10):
    time1=time.time()
    compute_Invar_nullspace(wght,reshelve=True)
    print('nullspace of weight',wght,'computed in time',hrs_min_sec(time.time()-time1),'\n')